<table>
  <tr>
    <td width="220" align="center" valign="middle">
      <picture>
        <source media="(prefers-color-scheme: dark)" srcset="https://console.anosys.ai/assets/icon/anosyslogod.svg">
        <source media="(prefers-color-scheme: light)" srcset="https://console.anosys.ai/assets/icon/anosyslogol.svg">
        <img src="https://console.anosys.ai/assets/icon/anosyslogol.svg" width="200" alt="AnoSys Logo">
      </picture>
    </td>
    <td valign="middle">
      <h1 style="margin:0;">LangChain Agentic - Observability Example</h1>
      <p style="margin:8px 0 0 0;">
        Interactive notebook demonstrating LangChain agentic workflows, multi-tool calling, and automatic tracing with AnoSys.
      </p>
    </td>
  </tr>
</table>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/github/Anosys-AI/anosys-sdk/blob/main/examples/anosys_langchain_agentic_poc.ipynb)

Welcome to the **AnoSys LangChain Agentic PoC**! This notebook demonstrates how to build an agentic environment using **LangChain** and gain deep visibility and automatic tracing into its behavior using **AnoSys Observability**.

### What you will learn:
1. How to initialize the AnoSys OpenAI Logger.
2. How to define custom tools using LangChain's `@tool` decorator.
3. How to bind tools to a ChatOpenAI model using modern LangChain tool-calling APIs (`bind_tools`).
4. How to implement a clean ReAct execution loop to handle multi-turn agent tool executions.
5. How AnoSys automatically captures traces for LLM calls and tool usages made by LangChain.


## 🛠️ Step 1: Installation

First, we need to install the required libraries. We will install:
- `anosys-sdk-openai` (for automatic OpenAI instrumentation and logging)
- `langchain-openai` and `langchain-core` (for the LangChain agentic workflow)

In [ ]:
!pip install -q -U anosys-sdk-openai langchain-openai langchain-core

## 🔑 Step 2: Configuration

To run this PoC, you will need two API keys:
1. **OpenAI API Key**: To power the completions and agent reasoning.
2. **AnoSys API Key**: To send traces to your observability dashboard.

We use Google Colab's secrets (`userdata`) to fetch the keys. Ensure you have stored them under the names `OPENAI_API_KEY` and `ANOSYS_API_KEY` in the Secrets tab (🔑 icon on the left).

In [ ]:
import os
try:
    from google.colab import userdata
    
    if "OPENAI_API_KEY" not in os.environ:
        os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    if "ANOSYS_API_KEY" not in os.environ:
        os.environ["ANOSYS_API_KEY"] = userdata.get('ANOSYS_API_KEY')
except ImportError:
    import getpass
    if "OPENAI_API_KEY" not in os.environ:
        os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key: ")
    if "ANOSYS_API_KEY" not in os.environ:
        os.environ["ANOSYS_API_KEY"] = getpass.getpass("Enter your AnoSys API Key: ")

## 🛰️ Step 3: Initialize AnoSys Observability

The `AnosysOpenAILogger` automatically instruments the OpenAI library using OpenTelemetry under the hood. Since LangChain's `ChatOpenAI` uses the standard OpenAI library, all agent LLM calls and tool integrations are automatically intercepted and sent to your AnoSys workspace.

In [ ]:
from anosys_sdk_openai import AnosysOpenAILogger

# Optional: Attach user context to correlate traces
def get_user_context():
    return {
        "session_id": "langchain-poc-001",
        "token": "langchain-demo-token"
    }

# Initialize the logger (do this once at startup)
AnosysOpenAILogger(get_user_context=get_user_context)

print("✅ AnoSys OpenAI Observability initialized!")


## 🤖 Step 4: Define Tools and Bind to the Model

Now we will build the agentic environment using modern LangChain tool-calling:
1. We will define two mock tools (`get_weather` and `search_flights`) using LangChain's `@tool` decorator.
2. We will initialize a `ChatOpenAI` LLM model and bind the tools to it via `bind_tools`.

In [ ]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

# --- Define Custom Tools ---

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    print(f"[Tool] Calling get_weather for {city}...")
    return f"The weather in {city} is sunny and 22°C."

@tool
def search_flights(origin: str, destination: str) -> str:
    """Search for flights between two cities."""
    print(f"[Tool] Calling search_flights from {origin} to {destination}...")
    return f"Found flights from {origin} to {destination}: Flight AN123 ($450, departing 10:00 AM)."

tools = [get_weather, search_flights]

# --- Initialize Model and Bind Tools ---

# Choose model and bind the tools
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
model_with_tools = llm.bind_tools(tools)

print("✅ LangChain model with tools is ready!")

## 🏃‍♂️ Step 5: Execute the Agentic Flow

We implement a clean ReAct execution loop. When the model requests a tool call, we execute the corresponding tool function and send the result back to the model as a `ToolMessage` to generate the final response.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

# Initialize message history
messages = [
    SystemMessage(content="You are a helpful assistant. Use tools to answer questions when necessary."),
    HumanMessage(content="I want to travel from London to Tokyo. Can you check the flights for me?")
]

print("Sending initial query to LLM...")
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Handle tool calls if returned by the LLM
if ai_msg.tool_calls:
    for tool_call in ai_msg.tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        
        # Dispatch and execute the tool
        if tool_name == "get_weather":
            tool_output = get_weather.invoke(tool_args)
        elif tool_name == "search_flights":
            tool_output = search_flights.invoke(tool_args)
        else:
            tool_output = f"Error: Tool {tool_name} not found."
            
        print(f"\n[Agent] Executed tool '{tool_name}' with args {tool_args}")
        print(f"[Agent] Tool Output: {tool_output}")
        
        # Append the tool message to history
        messages.append(ToolMessage(content=str(tool_output), tool_call_id=tool_call["id"]))
        
    # Get final answer from LLM with tool output in context
    print("\nSending tool output back to LLM...")
    final_ai_msg = model_with_tools.invoke(messages)
    
    print("\n--- Final Agent Response ---")
    print(final_ai_msg.content)
else:
    print("\n--- LLM Response (No tool call requested) ---")
    print(ai_msg.content)

print("\n✅ Trace captured! Head over to your AnoSys dashboard to view the agentic flow.")

## 📊 Step 6: Explore Your Traces in AnoSys

Go to the **AnoSys Console** to view the resulting trace.

### What you will observe in the trace:
1. **Parent LLM Call**: The agent's first execution asking to check flights.
2. **Tool Invocation**: The execution details of the `search_flights` tool, including inputs (`origin="London", destination="Tokyo"`) and outputs.
3. **Response Integration**: The final LLM completion incorporating the tool output.
4. **Token Usage**: Accurate metrics on prompts and completion tokens.

---
*Generated by Antigravity (AnoSys AI Coding Assistant)*